# Medical LLM Metacognition Benchmark — Analysis Notebook

This notebook reproduces the core analyses and figures for the benchmark.

It expects either:

- parsed outputs under `outputs/<model>/parsed_results.csv`, or
- zipped run outputs such as `pilot_gpt41nano_results.zip`.

Main outputs:
- overall metrics
- condition-level summaries
- confidence-by-evidence figure
- accuracy-by-evidence figure
- type-2 ROC
- confidence regressions
- exploratory model-comparison AUROC2 figure

In [ ]:
# Uncomment in Colab if needed:
# %pip install -q pandas numpy matplotlib scipy scikit-learn statsmodels

from pathlib import Path
import zipfile
import re
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import norm
from sklearn.metrics import roc_curve, auc
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "analysis" else NOTEBOOK_DIR

OUTPUTS_DIR = REPO_ROOT / "outputs"
FIGURES_DIR = REPO_ROOT / "figures"
RESULTS_DIR = REPO_ROOT / "results"

FIGURES_DIR.mkdir(exist_ok=True, parents=True)
RESULTS_DIR.mkdir(exist_ok=True, parents=True)

print("Repository root:", REPO_ROOT.resolve())

In [ ]:
def extract_timestamp(name: str):
    m = re.search(r"_(\d{8}_\d{6})\.", str(name))
    return m.group(1) if m else ""

def normalize_bool_series(s):
    if s.dtype == object:
        return s.replace({"True": True, "False": False, "true": True, "false": False})
    return s

def clean_results(df):
    df = df.copy()
    if "vignette_id" not in df.columns:
        if "vignette_id_x" in df.columns:
            df["vignette_id"] = df["vignette_id_x"]
        elif "vignette_id_y" in df.columns:
            df["vignette_id"] = df["vignette_id_y"]

    for col in ["parsed_ok", "forced_choice_accuracy_applicable", "choice_correct",
                "more_information_needed", "more_info_correct", "expected_more_information_needed"]:
        if col in df.columns:
            df[col] = normalize_bool_series(df[col])

    df["confidence"] = pd.to_numeric(df["confidence"], errors="coerce")
    df["confidence_prob"] = df["confidence"] / 100.0
    return df

def load_parsed_from_zip(zip_path):
    zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        parsed_files = [n for n in zf.namelist() if "parsed_results_" in n and n.endswith(".csv")]
        if not parsed_files:
            raise FileNotFoundError(f"No parsed_results_*.csv found in {zip_path}")
        parsed_name = sorted(parsed_files, key=extract_timestamp)[-1]
        with zf.open(parsed_name) as f:
            df = pd.read_csv(f)
    df = clean_results(df)
    df.attrs["source_file"] = parsed_name
    return df

def load_model_results(model_name, zip_path=None):
    if zip_path:
        df = load_parsed_from_zip(zip_path)
    else:
        path = OUTPUTS_DIR / model_name / "parsed_results.csv"
        if not path.exists():
            raise FileNotFoundError(f"Missing {path}; provide zip_path instead.")
        df = clean_results(pd.read_csv(path))
        df.attrs["source_file"] = str(path)
    df.attrs["model_name"] = model_name
    return df

def forced_choice_df(df):
    return df[
        (df["parsed_ok"] == True) &
        (df["forced_choice_accuracy_applicable"] == True) &
        (df["choice_correct"].notna()) &
        (df["confidence"].notna())
    ].copy()

def expected_calibration_error(y_true, p_pred, n_bins=10):
    y_true = np.asarray(y_true, dtype=float)
    p_pred = np.asarray(p_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(p_pred)
    y_true, p_pred = y_true[mask], p_pred[mask]
    if len(y_true) == 0:
        return np.nan
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        in_bin = (p_pred >= lo) & ((p_pred < hi) if i < n_bins - 1 else (p_pred <= hi))
        if in_bin.any():
            ece += in_bin.mean() * abs(y_true[in_bin].mean() - p_pred[in_bin].mean())
    return float(ece)

def type2_roc(confidence, correctness):
    conf = np.asarray(confidence, dtype=float)
    corr = np.asarray(correctness, dtype=bool)
    pos = conf[corr]
    neg = conf[~corr]

    if len(pos) == 0 or len(neg) == 0:
        return np.array([0, 1]), np.array([0, 1]), np.nan

    thresholds = np.r_[np.inf, np.sort(np.unique(conf))[::-1], -np.inf]
    tpr, fpr = [], []
    for t in thresholds:
        tpr.append(np.mean(pos >= t))
        fpr.append(np.mean(neg >= t))

    fpr = np.asarray(fpr, dtype=float)
    tpr = np.asarray(tpr, dtype=float)
    order = np.argsort(fpr)
    fpr, tpr = fpr[order], tpr[order]
    auroc2 = np.trapezoid(tpr, fpr)
    return fpr, tpr, auroc2

def overall_metrics(df):
    fc = forced_choice_df(df)
    y = fc["choice_correct"].astype(int).values
    p = fc["confidence_prob"].astype(float).values
    _, _, auroc2 = type2_roc(p, y.astype(bool))
    return {
        "total_trials": int(len(df)),
        "forced_choice_trials": int(len(fc)),
        "parsed_rate": float(df["parsed_ok"].mean()),
        "forced_choice_accuracy": float(np.mean(y)),
        "mean_confidence": float(np.mean(p)),
        "brier_score": float(np.mean((p - y) ** 2)),
        "ece_10_bins": expected_calibration_error(y, p),
        "auroc2": auroc2,
        "more_information_needed_accuracy": float(df["more_info_correct"].mean()) if "more_info_correct" in df.columns else np.nan,
    }

## Load primary model

Set `PRIMARY_ZIP_PATH` if using a zipped run output. Otherwise the notebook will look for `outputs/gpt-4.1-nano/parsed_results.csv`.

In [ ]:
PRIMARY_MODEL = "gpt-4.1-nano"
PRIMARY_ZIP_PATH = None

# Example:
# PRIMARY_ZIP_PATH = "/content/pilot_gpt41nano_results.zip"

# Auto-detect common zip names in current folder or repository root.
if PRIMARY_ZIP_PATH is None:
    candidates = list(Path(".").glob("pilot_gpt41nano_results.zip")) + list(REPO_ROOT.glob("pilot_gpt41nano_results.zip"))
    if candidates:
        PRIMARY_ZIP_PATH = str(candidates[0])

primary = load_model_results(PRIMARY_MODEL, zip_path=PRIMARY_ZIP_PATH)
print(primary.shape)
print(primary.attrs.get("source_file"))
primary.head()

## Overall metrics

In [ ]:
primary_metrics = pd.DataFrame([overall_metrics(primary)])
display(primary_metrics)
primary_metrics.to_csv(RESULTS_DIR / "primary_overall_metrics.csv", index=False)

## Condition-level summary

In [ ]:
condition_summary = (
    primary.groupby(["evidence_level", "information_quality"], as_index=False)
    .agg(
        n=("trial_id", "count"),
        parsed_rate=("parsed_ok", "mean"),
        mean_confidence=("confidence_prob", "mean"),
        empirical_accuracy=("choice_correct", "mean"),
        more_info_rate=("more_information_needed", "mean"),
        more_info_accuracy=("more_info_correct", "mean"),
    )
)
condition_summary["overconfidence"] = condition_summary["mean_confidence"] - condition_summary["empirical_accuracy"]
display(condition_summary)
condition_summary.to_csv(RESULTS_DIR / "primary_condition_summary.csv", index=False)

## Figure 2 — Confidence as a function of evidence level by information quality

In [ ]:
levels_all = ["strong_DRCI", "moderate_DRCI", "equivocal", "moderate_AT_NCD", "strong_AT_NCD"]
label_map = {
    "strong_DRCI": "Strong DRCI",
    "moderate_DRCI": "Moderate DRCI",
    "equivocal": "Equivocal",
    "moderate_AT_NCD": "Moderate AT-NCD",
    "strong_AT_NCD": "Strong AT-NCD",
}

conf_df = primary[(primary["parsed_ok"] == True) & (primary["confidence"].notna())].copy()

conf_strat = (
    conf_df.groupby(["evidence_level", "information_quality"], as_index=False)
    .agg(
        mean_confidence=("confidence", "mean"),
        sd_confidence=("confidence", "std"),
        n=("trial_id", "count")
    )
)
conf_strat["se_confidence"] = conf_strat["sd_confidence"] / np.sqrt(conf_strat["n"])
conf_strat["evidence_level"] = pd.Categorical(conf_strat["evidence_level"], categories=levels_all, ordered=True)
conf_strat = conf_strat.sort_values(["information_quality", "evidence_level"])

plt.figure(figsize=(8.8, 5.4))
for info in ["clear", "conflicting", "missing"]:
    sub = conf_strat[conf_strat["information_quality"] == info].copy().sort_values("evidence_level")
    x_labels = [label_map[x] for x in sub["evidence_level"].astype(str)]
    plt.errorbar(x_labels, sub["mean_confidence"], yerr=sub["se_confidence"], marker="o", capsize=4, label=info.capitalize())

plt.ylabel("Mean confidence (0–100)")
plt.xlabel("Evidence level")
plt.title("Confidence as a function of evidence level by information quality")
plt.legend(title="Information quality")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "figure2_confidence_vs_evidence_by_information_quality.png", dpi=300, bbox_inches="tight")
plt.savefig(FIGURES_DIR / "figure2_confidence_vs_evidence_by_information_quality.pdf", bbox_inches="tight")
plt.show()

conf_strat.to_csv(RESULTS_DIR / "figure2_confidence_vs_evidence_by_information_quality.csv", index=False)

## Figure 3 — Accuracy as a function of evidence level

In [ ]:
acc_df = primary[
    (primary["parsed_ok"] == True) &
    (primary["forced_choice_accuracy_applicable"] == True) &
    (primary["choice_correct"].notna())
].copy()

acc_summary = acc_df.groupby("evidence_level", as_index=False).agg(empirical_accuracy=("choice_correct", "mean"), n=("trial_id", "count"))

grid = pd.DataFrame({"evidence_level": levels_all})
acc_plot = grid.merge(acc_summary, on="evidence_level", how="left")
acc_plot["evidence_level"] = pd.Categorical(acc_plot["evidence_level"], categories=levels_all, ordered=True)
acc_plot = acc_plot.sort_values("evidence_level").reset_index(drop=True)
acc_plot["evidence_label"] = acc_plot["evidence_level"].astype(str).map(label_map)

x = np.arange(len(acc_plot))
plt.figure(figsize=(8.8, 5.2))
plt.plot(x, acc_plot["empirical_accuracy"], marker="o")

for i, row in acc_plot.iterrows():
    if pd.notna(row["empirical_accuracy"]):
        plt.text(i, row["empirical_accuracy"] + 0.035, f"n={int(row['n'])}", ha="center", fontsize=9)

eq_idx = acc_plot.index[acc_plot["evidence_level"] == "equivocal"][0]
plt.text(eq_idx, 0.08, "Accuracy\nnot applicable", ha="center", fontsize=9)

plt.xticks(x, acc_plot["evidence_label"], rotation=20)
plt.ylim(0, 1.05)
plt.ylabel("Empirical accuracy")
plt.xlabel("Evidence level")
plt.title("Accuracy as a function of evidence level")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "figure3_accuracy_vs_evidence.png", dpi=300, bbox_inches="tight")
plt.savefig(FIGURES_DIR / "figure3_accuracy_vs_evidence.pdf", bbox_inches="tight")
plt.show()

acc_plot.to_csv(RESULTS_DIR / "figure3_accuracy_vs_evidence.csv", index=False)

## Figure 4 — Type-2 ROC for primary model

In [ ]:
fc = forced_choice_df(primary)
fpr, tpr, auroc2 = type2_roc(fc["confidence_prob"], fc["choice_correct"].astype(bool))

plt.figure(figsize=(5.4, 5.0))
plt.plot(fpr, tpr, marker="o", label=f"AUROC2 = {auroc2:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.xlabel("Type-2 false positive rate")
plt.ylabel("Type-2 hit rate")
plt.title("Type-2 ROC")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "figure4_type2_ROC.png", dpi=300, bbox_inches="tight")
plt.savefig(FIGURES_DIR / "figure4_type2_ROC.pdf", bbox_inches="tight")
plt.show()

## Confidence regression analyses

In [ ]:
conf_df = primary[(primary["parsed_ok"] == True) & (primary["confidence"].notna())].copy()
conf_df["evidence_distance"] = conf_df["evidence_score"].abs()

fit_conf = smf.ols(
    "confidence ~ evidence_distance + C(information_quality) + C(prompt_id)",
    data=conf_df
).fit()

print(fit_conf.summary())

with open(RESULTS_DIR / "confidence_model_summary.txt", "w") as f:
    f.write(str(fit_conf.summary()))

In [ ]:
conf_choice_df = forced_choice_df(primary)
conf_choice_df["evidence_distance"] = conf_choice_df["evidence_score"].abs()
conf_choice_df["choice_correct_int"] = conf_choice_df["choice_correct"].astype(int)

fit_conf_correct = smf.ols(
    "confidence ~ choice_correct_int + evidence_distance + C(information_quality) + C(prompt_id)",
    data=conf_choice_df
).fit(
    cov_type="cluster",
    cov_kwds={"groups": conf_choice_df["vignette_id"]}
)

print(fit_conf_correct.summary())

with open(RESULTS_DIR / "confidence_correctness_model_summary.txt", "w") as f:
    f.write(str(fit_conf_correct.summary()))

## Exploratory cross-model comparison

In [ ]:
MODEL_ZIPS = {
    "gpt-4.1-nano": None,
    "gpt-4.1-mini": None,
    "gpt-5-nano": None,
    "gpt-5": None,
    "gpt-5.5": None,
}

auto_zip_names = {
    "gpt-4.1-nano": "pilot_gpt41nano_results.zip",
    "gpt-4.1-mini": "pilot_gpt41mini_results.zip",
    "gpt-5-nano": "pilot_gpt5nano_results.zip",
    "gpt-5": "pilot_gpt5_results_final.zip",
    "gpt-5.5": "pilot_gpt5_5_results.zip",
}

for model, fname in auto_zip_names.items():
    candidates = list(Path(".").glob(fname)) + list(REPO_ROOT.glob(fname))
    if candidates:
        MODEL_ZIPS[model] = str(candidates[0])

model_results = {}
for model, zpath in MODEL_ZIPS.items():
    try:
        model_results[model] = load_model_results(model, zip_path=zpath)
        print(f"Loaded {model}: {model_results[model].shape}")
    except Exception as e:
        print(f"Skipping {model}: {e}")

In [ ]:
summary_rows = []
roc_data = {}

for model, df in model_results.items():
    fc_model = forced_choice_df(df)
    if len(fc_model) == 0:
        continue
    y = fc_model["choice_correct"].astype(int).values
    p = fc_model["confidence_prob"].astype(float).values
    fpr, tpr, auroc2 = type2_roc(p, y.astype(bool))
    roc_data[model] = {"fpr": fpr, "tpr": tpr, "auroc2": auroc2, "n": len(fc_model)}
    summary_rows.append({
        "model": model,
        "n_total_rows": len(df),
        "parsed_rate": df["parsed_ok"].mean(),
        "n_forced_choice_parsed": len(fc_model),
        "accuracy": y.mean(),
        "mean_confidence": p.mean(),
        "brier_score": np.mean((p - y) ** 2),
        "ece_10_bins": expected_calibration_error(y, p),
        "auroc2": auroc2,
        "more_info_accuracy": df["more_info_correct"].mean() if "more_info_correct" in df.columns else np.nan,
    })

comparison = pd.DataFrame(summary_rows)
order = ["gpt-4.1-nano", "gpt-4.1-mini", "gpt-5-nano", "gpt-5", "gpt-5.5"]
if len(comparison):
    comparison["model"] = pd.Categorical(comparison["model"], categories=order, ordered=True)
    comparison = comparison.sort_values("model").reset_index(drop=True)
    display(comparison)
    comparison.to_csv(RESULTS_DIR / "model_comparison_summary.csv", index=False)

In [ ]:
if len(comparison):
    bar_df = comparison[comparison["auroc2"].notna()].copy()

    plt.figure(figsize=(8.5, 5.2))
    bars = plt.bar(bar_df["model"].astype(str), bar_df["auroc2"])

    for bar, (_, row) in zip(bars, bar_df.iterrows()):
        x = bar.get_x() + bar.get_width() / 2
        y = row["auroc2"]
        plt.text(x, y + 0.015, f"{y:.3f}", ha="center", va="bottom", fontsize=10)
        plt.text(x, max(0.03, y * 0.08), f"n={int(row['n_forced_choice_parsed'])}", ha="center", va="bottom", fontsize=9)

    plt.ylim(0, 1.05)
    plt.ylabel("AUROC2")
    plt.xlabel("Model")
    plt.title("Metacognitive sensitivity across models")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "figure5_auroc2_model_comparison.png", dpi=300, bbox_inches="tight")
    plt.savefig(FIGURES_DIR / "figure5_auroc2_model_comparison.pdf", bbox_inches="tight")
    plt.show()

## Export note

Generated outputs are saved in:

- `figures/`
- `results/`

Copy final figures into `manuscript/figures/` before compiling the LaTeX manuscript.